In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 03 — PDF / Document Input

Send PDFs to **`claude-opus-4-8`** as **base64 document blocks** through the Agent Platform to summarize them, answer questions, and extract structured data. Claude reads **both the text and the visual layout** of the pages (tables, columns, figures).

This builds on **Modules 00–02** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]" reportlab

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Create a self-contained sample PDF

To keep the notebook self-contained, we **generate** a small sample PDF locally (no external URLs, no copyrighted content) and save it under the existing `assets/` folder. It includes a title, a couple of paragraphs, and a small table so there is structured data to extract.

In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

os.makedirs("assets", exist_ok=True)
PDF_PATH = "assets/sample_doc.pdf"

styles = getSampleStyleSheet()
story = [
    Paragraph("Quarterly Operations Report", styles["Title"]),
    Spacer(1, 12),
    Paragraph(
        "This report summarizes regional sales performance for the most recent "
        "quarter. Figures are shown in thousands of units and are illustrative "
        "sample data generated for this tutorial.",
        styles["BodyText"],
    ),
    Spacer(1, 8),
    Paragraph(
        "The Central region led on volume, while the West region showed the "
        "strongest quarter-over-quarter growth. See the table below for details.",
        styles["BodyText"],
    ),
    Spacer(1, 16),
]

table_data = [
    ["Region", "Units (thousands)"],
    ["North", "45"],
    ["South", "38"],
    ["East", "52"],
    ["West", "61"],
    ["Central", "74"],
]
table = Table(table_data, hAlign="LEFT")
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#4C72B0")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 10),
    ("BOTTOMPADDING", (0, 0), (-1, 0), 6),
]))
story.append(table)

SimpleDocTemplate(PDF_PATH, pagesize=letter).build(story)
print(f"✅ Saved sample PDF to: {os.path.abspath(PDF_PATH)}")

## Part D — Send the PDF (base64 document block)

For document input, a message's `content` is a **list** that mixes a **document block** and a **text block**. The document block carries a **base64** `source` with `media_type` set to **`application/pdf`**; the text block is your instruction.

In [ ]:
import base64

def pdf_to_base64(path: str) -> str:
    """Read a local PDF file and return its base64-encoded data."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()

pdf_b64 = pdf_to_base64(PDF_PATH)

message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": pdf_b64,
                },
            },
            {"type": "text", "text": "Summarize this document in 3 bullet points."},
        ],
    }],
)
print(message.content[0].text)

## Part E — Extraction / Q&A + notes

Documents support **targeted extraction**, not just summarization — you can ask for a specific value, and Claude reads the table to answer.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": pdf_b64,
                },
            },
            {"type": "text", "text": "What is the total units across all regions in the table? Answer in one line."},
        ],
    }],
)
print(message.content[0].text)

### Notes

- Claude sees **both the extracted text and the rendered pages**, so layout, tables, and figures are understood — not just the raw text stream.
- **Base64 is the universal path** on the Agent Platform for document input.
- There are **page-count and size limits**, and **scanned PDFs** (image-only) rely on the visual read of each page.
- See the Anthropic **PDF support** docs (https://docs.claude.com/en/docs/build-with-claude/pdf-support) and confirm the current limits there, as they may change.

## Part F — Recap

- Document input uses a **content list** mixing a **document block** (base64 `source` + `media_type` `application/pdf`) and a **text block**.
- We generated a **self-contained** sample PDF locally into `assets/` — no external assets needed.
- Claude reads **text + visual layout**, enabling summarization *and* targeted extraction from tables.
- Mind page-count/size limits; scanned PDFs rely on the visual read — confirm current limits in the PDF support docs.

**Next: Module 04 — Function calling (tool use).**